## Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import torch
from torch.utils.data import Dataset
from transformers import (pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,)
from sklearn.metrics import classification_report, confusion_matrix
import json

## Config

In [2]:
#### Config ####
MODEL_NAME = "xlm-roberta-base"
TRAIN_PATH   = Path("../data/splits/train.csv")
VAL_PATH     = Path("../data/splits/val.csv")
TEST_PATH    = Path("../data/splits/test.csv")
MODEL_OUTPUT = Path("../models/sentiment-v1")
RESULTS_PATH = Path("../results/eval_results.txt")

### Data loading and inspection

In [3]:
df = pd.read_excel(r"D:\Studying\Machine-Learning\ML-projects\gr-reviews-model\data\raw\Sentiment-analysis.xlsx")
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

In [4]:
print(f"Train set has: {len(train_df)} rows")
print(f"Validation set has: {len(val_df)} rows")
print(f"Test set has: {len(test_df)} rows")

print("\nLabel distribution in training set:")
print(train_df["label"].value_counts())

print("\nSample reviews:")
train_df[["text", "label"]].sample(5)

Train set has: 160 rows
Validation set has: 20 rows
Test set has: 20 rows

Label distribution in training set:
label
positive    67
negative    63
neutral     30
Name: count, dtype: int64

Sample reviews:


,text,label
9,Θεωρώ ότι είναι από τις καλύτερες κονσόλες αυτ...,positive
63,Παρά πολύ καλή μηχανή. Πολύ ευχαριστημένος!!,positive
73,Το κινητό υπερθερμαίνεται χωρίς αιτία\nΌταν δε...,negative
10,Αν και καλή δεν νομίζω να την ξανά αγόραζα. Υπ...,neutral
46,"Είναι αθόρυβα το αριστερό και δεξί κλικ, κατά ...",negative


In [5]:
print(df.head(2))

                                                text     label
0  Έρχομαι από galaxy 5 , δυστυχώς το galaxy 7 ει...  negative
1  Άνετα και πολύ όμορφα παπούτσια. Δεν μπορούσαν...  negative


## Testing pipeline

In [33]:
classifier = pipeline("sentiment-analysis")

classifier(
    ["Έρχομαι από galaxy 5 , δυστυχώς το galaxy 7 ειναι κακοφτιαγμένο \n" \
"και στο μπροστα μέρος πολύ χαμηλό  με αποτέλεσμα να χτυπάει το δάχτυλο \n" \
" λόγο πολύ μικρού χώρου .Δηλαδή δεν ξέρω τι σκεφτόταν ο σχεδιαστή \n" \
"Update : Κόπηκαν κιόλας σε διάφορα σημεία μετά από μόλις 3 μήνες με ελάχιστη χρήση. \n" \
" Πρώτη φορά μου συμβαίνει κάτι τέτοιο τόσο σύντομα",

"Είναι τέλειο και αξίζει τα χρήματα του αν θες να επενδύσεις σε καλό κινητό. \n "
"Προσωπικά πήγα από iPhone 11 σε iPhone 16 pro max και πραγματικά καμία σχέση. Θα το ευχαριστηθείτε",
    ]
)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.8464093208312988},
 {'label': 'POSITIVE', 'score': 0.5289077162742615}]

## Tokenizer

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
test = df["text"][0]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

d:\Studying\Machine-Learning\ML-projects\gr-reviews-model\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--xlm-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
tokens = tokenizer(test, return_tensors="pt")

In [ ]:
print(f"Original text: {test}")
print(f"Token IDs:     {tokens['input_ids'][0].tolist()}")
print(f"Decoded back:  {tokenizer.decode(tokens['input_ids'][0])}")
print(f"\nTokenizer loaded successfully.")

Original text: Έρχομαι από galaxy 5 , δυστυχώς το galaxy 7 ειναι κακοφτιαγμένο και στο μπροστα μέρος πολύ χαμηλό με αποτέλεσμα να χτυπάει το δάχτυλο λόγο πολύ μικρού χώρου .
Δηλαδή δεν ξέρω τι σκεφτόταν ο σχεδιαστής

Update : Κόπηκαν κιόλας σε διάφορα σημεία μετά από μόλις 3 μήνες με ελάχιστη χρήση . Πρώτη φορά μου συμβαίνει κάτι τέτοιο τόσο σύντομα
Token IDs:     [0, 19905, 55149, 70878, 679, 225495, 190, 6, 4, 209570, 374, 225495, 361, 73729, 161443, 4639, 6623, 17934, 48690, 215, 1007, 23049, 9249, 16143, 48639, 8191, 187518, 558, 49677, 386, 142729, 48787, 374, 6, 63244, 2088, 73583, 14292, 54935, 8191, 229413, 187477, 6, 5, 217912, 1604, 141743, 17124, 182628, 45834, 1038, 192654, 44561, 235, 28641, 152, 151611, 37483, 75273, 10126, 97494, 235, 962, 169667, 149455, 9930, 679, 85834, 138, 54778, 558, 47972, 1206, 44656, 20189, 45041, 6, 5, 133399, 169548, 29059, 4556, 127540, 23503, 105651, 24316, 204763, 2]
Decoded back:  <s> Έρχομαι από galaxy 5 , δυστυχώς το galaxy 7 ειναι κακοφ

In [11]:
test

'Έρχομαι από galaxy 5 , δυστυχώς το galaxy 7 ειναι κακοφτιαγμένο και στο μπροστα μέρος πολύ χαμηλό με αποτέλεσμα να χτυπάει το δάχτυλο λόγο πολύ μικρού χώρου .\nΔηλαδή δεν ξέρω τι σκεφτόταν ο σχεδιαστής\n\nUpdate : Κόπηκαν κιόλας σε διάφορα σημεία μετά από μόλις 3 μήνες με ελάχιστη χρήση . Πρώτη φορά μου συμβαίνει κάτι τέτοιο τόσο σύντομα'